# 01 · 模型与提示词（Model I/O）

本章聚焦 LangChain 最底层的抽象 **Model I/O**：如何把「提示词」喂给「模型」，再把「模型输出」解析成想要的格式。

建议逐个单元格运行（Shift+Enter）。运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已执行 `pip install -r requirements.txt`
- 已复制 `.env.example` 为 `.env` 并填入 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/01-models-and-prompts.md`。


## 0. 环境检查

先确认能加载 `.env` 中的 API Key。


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


## 1. 最基础的对话调用

`ChatOpenAI` 是对话模型统一的入口。直接调用 `.invoke('...')` 即可得到回复。


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))
print(llm.invoke('你好，介绍一下你自己。').content)


## 2. 用 ChatPromptTemplate 动态构造多角色提示词

`ChatPromptTemplate` 把可变部分参数化，支持 system / human 多角色消息。用 `.format()` 打印检查变量替换是否正确。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个{role}，请用专业且简洁的语言回答。'),
    ('human', '{question}'),
])
chain = prompt | llm | StrOutputParser()
print(chain.invoke({'role': 'Python 专家', 'question': '什么是装饰器？'}))


## 3. 用 PromptTemplate 做单段文本模板

`PromptTemplate` 适合简单的「一段字符串 + 变量」场景，例如翻译一句话。


In [ ]:
from langchain_core.prompts import PromptTemplate

tpl = PromptTemplate.from_template('把下面句子翻译成英文：{sentence}')
print((tpl | llm | StrOutputParser()).invoke({'sentence': '今天天气真好'}))


## 4. 结构化输出：PydanticOutputParser

用 Pydantic 模型约束返回字段，让模型输出可被程序直接消费。`.partial(format_instructions=...)` 把格式说明注入 system 消息。


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Translation(BaseModel):
    language: str = Field(description='目标语言名称')
    text: str = Field(description='翻译后的文本')

parser = PydanticOutputParser(pydantic_object=Translation)
struct_prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个翻译器。\n{format_instructions}'),
    ('human', '把下面内容翻译成{target_language}：\n{text}'),
]).partial(format_instructions=parser.get_format_instructions())

struct_chain = struct_prompt | llm | parser
result = struct_chain.invoke({'target_language': '英语', 'text': '你好，世界'})
print(f'language={result.language}, text={result.text}')


## 5. 下一步

- 把多个组件串成链（LCEL）：运行 `examples/02_chains.py`，参见 `docs/02-chains.md`
- 多轮记忆：运行 `examples/03_memory.py`，参见 `docs/03-memory.md`

**常见坑**：模板变量名拼错会在运行时才报错；先用 `.format()` 打印检查。不要把密钥或隐私塞进提示词。
